# Construction Site Safety — Dataset Preparation

I used a hybrid dataset preparation approach by combining publicly available PPE datasets from Kaggle and Roboflow with additional custom images collected from construction-site scenes and carefully selected web sources such as Google Images.


## 1 · Overview

| Property | Value |
|---|---|
| Total images after cleaning | **~3,400** |
| Annotation format | YOLO (normalised `cx cy w h`) |
| Annotation tool | Roboflow |
| Number of classes | 5 |
| Train / Val / Test split | 70% / 20% / 10% |

This notebook handles the full data pipeline:
1. Install dependencies
2. Download and fork datasets from kaggle and roboflow
3. Remap all labels to a unified schema
4. Merge into a single dataset
5. De-duplicate images
6. Analyse and visualise class distribution
8. Export final dataset ready for YOLOv8 training

**Unified label schema:**
| ID | Class | Description |
|---|---|---|
| 0 | `person` | Any visible worker |
| 1 | `helmet` | Worker wearing a hard hat |
| 2 | `no-helmet` | Worker without head protection |
| 3 | `vest` | Worker wearing a high-vis vest |
| 4 | `no-vest` | Worker without a high-vis vest |

# 2· Dataset Colllection

## Dataset Sources


| Source                                      | Images | License    | Usage |
|---------------------------------------------|-------:|------------|-------|
|[Construction Safety Dataset (Roboflow 100)](https://universe.roboflow.com/roboflow-100/construction-safety-gsnvb) | ~1200 | CC BY 4.0  | Primary base dataset  and fork this dataset|
| [Construction Site Safety Dataset (Kaggle)](https://www.kaggle.com/datasets/snehilsanyal/construction-site-safety-image-dataset-roboflow)     | ~2800   | CC BY 4.0  | Used as main dataset |
| [No Helmet-No Vest Computer Vision Dataset (Roboflow)](https://universe.roboflow.com/ai-thesis-s3bl1/no-helmet-no-vest)                          | ~700    | CC BY 4.0       | Volaton Detection |
| [No Helmet-No Vest Computer Vision Dataset (Roboflow)](https://universe.roboflow.com/ai-thesis-s3bl1/no-helmet-no-vest)                          | ~700    | CC BY 4.0       | Volaton Detection |
| [Construction-PPE Dataset](https://docs.ultralytics.com/datasets/detect/construction-ppe/#applications)                          | ~1400    | CC BY 4.0       | Increase the Diversity |
| Unsplash / Pexels                           | ~80    | CC0        | Custom addition (lighting and environment variation) |

---

## 3. Balancing Strategy Per Class

| Class | Original | Target | Strategy |
|---|---|---|---|
| `person` | 3,522 | ~2,200 | **Undersampled** — randomly removed images where this class dominates |
| `hard_hat` | 2,653 | ~2,200 | **Used as-is** — within acceptable range of target |
| `no_hard_hat` | 1,904 | ~2,200 | **Augmented** — 1.15x multiplier via Albumentations |
| `safety_vest` | 1,404 | ~2,200 | **Augmented 2x** — most underrepresented class |
| `no_safety_vest` | 2,882 | ~2,200 | **Used as-is** — within acceptable range |

**Excluded classes:**  
`machinery` `vehicle` `boots` and `gloves` were excluded — outside the scope of PPE compliance detection and this system decided to detect safety vest and hard had as the main safey equipments.

## 4. Dataset Diversity

| Dimension | Coverage |
|---|---|
| **Environments** | Outdoor construction sites (open lots, building frames, scaffolding) and indoor (warehouses, basement construction) |
| **Lighting** | Bright daylight, overcast, shadow, artificial indoor lighting |
| **Worker scale** | Close-range and distant workers (small bounding boxes reinforced via `mosaic=1.0` during training) |
| **Crowd size** | Single isolated workers and multi-worker group scenes |
| **PPE variety** | Yellow, white, orange hard hats; yellow, orange, green hi-vis vests |
| **Safe vs unsafe** | Balanced mix of compliant and violation scenes |

## 5. Annotation Approach

### Tool
All annotations were created or verified using the **Roboflow** annotation interface, which exports directly to YOLO format.

### Format
YOLO bounding box format — one `.txt` file per image, one row per object:
```
<class_id> <cx> <cy> <width> <height>
```
All values normalised to `[0, 1]` relative to image dimensions.

### Class remapping applied at export
```
Hardhat        ->  hard_hat
NO-Hardhat     ->  no_hard_hat
Safety Vest    ->  safety_vest
NO-Safety Vest ->  no_safety_vest
Person         ->  person
```

### positive + negative classes for hardhat and the vest
Both `hard_hat` and `no_hard_hat` are labelled as separate classes (and similarly for vests).  
This is a deliberate design choice — YOLO detects *objects*, not absence. An explicit `no_hard_hat` class gives the model a positive detection target for the violation, rather than relying on the absence of `hard_hat` which fails for occluded or distant workers.

### Dataset Download
By using Roboflow the dataset was downloaded in YOLOV8 format.

## 6. Dataset Cleaning Pipeline

A dedicated cleaning notebook (`data/data_clean.py`) was executed during the training within the notebook.  
The original dataset was **never modified** — the pipeline writes a new `_cleaned` copy.

| Check | Detection method | Action |
|---|---|---|
| Corrupt images | PIL `.verify()` + OpenCV `imread` returning None | Skip image + label |
| Missing label files | File existence check | Skip image |
| Empty label files | File content length | Skip image |
| Near-duplicate frames | Perceptual hash (pHash), distance <= 8 | Keep first, skip rest |
| Extreme aspect ratios (> 5:1) | Width/height ratio | Keep but log |
| Wrong column count in label row | Split length != 5 | Drop that row |
| Non-numeric label values | float() parse failure | Drop that row |
| Class ID out of range [0,4] | Integer range check | Drop that row |
| Coordinates outside [0,1] | Value range check | Auto-clamp to valid range |
| Tiny bounding boxes (area < 0.05%) | bw x bh < 0.0005 | Drop that row |
| Images with no valid rows remaining | Post-fix row count | Skip image |

## 7. Train / Val / Test Split

| Split | Proportion | Purpose |
|---|---|---|
| Train | 70% | Model training |
| Validation | 20% | Hyperparameter tuning, early stopping |
| Test | 10% | Final held-out evaluation — never seen during training |

Splits were created using Roboflow's built-in split generator.